# Coordinate Routine Validation

This notebook tests the SRT controller's coordinate routines against astropy to verify accuracy.

In [ ]:
# Install astropy if needed
# !pip install astropy

In [ ]:
import math
from datetime import datetime, timezone

# Astropy imports
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_sun, get_body
from astropy.time import Time
from astropy import units as u
import numpy as np

## Copy of SRT Coordinate Routines

These are copied from `esp32_controller/coordinates.py` and adapted for standard Python.

In [ ]:
def julian_date(year, month, day, hour, minute, second):
    """Calculate Julian Date from calendar date/time (UTC)"""
    if month <= 2:
        year -= 1
        month += 12

    A = int(year / 100)
    B = 2 - A + int(A / 4)

    jd = int(365.25 * (year + 4716)) + int(30.6001 * (month + 1)) + day + B - 1524.5
    jd += (hour + minute / 60 + second / 3600) / 24

    return jd


def gmst(jd):
    """Calculate Greenwich Mean Sidereal Time in hours from Julian Date"""
    T = (jd - 2451545.0) / 36525.0

    # GMST at 0h UT in seconds
    gmst_sec = 24110.54841 + 8640184.812866 * T + 0.093104 * T**2 - 6.2e-6 * T**3

    # Add rotation since 0h UT
    gmst_sec += 86400 * 1.00273790935 * ((jd - 0.5) % 1)

    # Convert to hours and normalize to 0-24
    gmst_hours = (gmst_sec / 3600) % 24

    return gmst_hours


def local_sidereal_time(jd, longitude):
    """Calculate Local Sidereal Time in hours"""
    lst = gmst(jd) + longitude / 15.0
    return lst % 24

In [ ]:
def get_sun_position(dt):
    """
    Calculate the Sun's RA/Dec for a given datetime.
    Based on Meeus "Astronomical Algorithms" - accurate to ~0.01 degree.
    """
    jd = julian_date(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second)

    # Julian centuries since J2000.0
    T = (jd - 2451545.0) / 36525.0

    # Geometric mean longitude of the Sun (degrees)
    L0 = (280.46646 + 36000.76983 * T + 0.0003032 * T**2) % 360

    # Mean anomaly of the Sun (degrees)
    M = (357.52911 + 35999.05029 * T - 0.0001537 * T**2) % 360
    M_rad = math.radians(M)

    # Equation of center (degrees)
    C = ((1.914602 - 0.004817 * T - 0.000014 * T**2) * math.sin(M_rad) +
         (0.019993 - 0.000101 * T) * math.sin(2 * M_rad) +
         0.000289 * math.sin(3 * M_rad))

    # Sun's true longitude (degrees)
    sun_lon = L0 + C

    # Apparent longitude (corrected for nutation and aberration)
    omega = 125.04 - 1934.136 * T
    sun_lon_apparent = sun_lon - 0.00569 - 0.00478 * math.sin(math.radians(omega))
    sun_lon_rad = math.radians(sun_lon_apparent)

    # Mean obliquity of the ecliptic
    eps0 = 23.439291 - 0.0130042 * T - 0.00000016 * T**2 + 0.000000504 * T**3

    # Corrected obliquity
    eps = eps0 + 0.00256 * math.cos(math.radians(omega))
    eps_rad = math.radians(eps)

    # Convert ecliptic to equatorial coordinates
    ra_rad = math.atan2(
        math.cos(eps_rad) * math.sin(sun_lon_rad),
        math.cos(sun_lon_rad)
    )
    dec_rad = math.asin(math.sin(eps_rad) * math.sin(sun_lon_rad))

    ra_hours = (math.degrees(ra_rad) / 15) % 24
    dec_deg = math.degrees(dec_rad)

    return ra_hours, dec_deg

In [ ]:
def get_moon_position(dt):
    """
    Calculate the Moon's RA/Dec for a given datetime.
    Based on Meeus "Astronomical Algorithms" Ch. 47 - accurate to ~0.3 degree.
    """
    jd = julian_date(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second)

    # Julian centuries since J2000.0
    T = (jd - 2451545.0) / 36525.0

    # Moon's mean longitude (degrees)
    Lp = (218.3164477 + 481267.88123421 * T - 0.0015786 * T**2 +
          T**3 / 538841 - T**4 / 65194000) % 360

    # Moon's mean elongation (degrees)
    D = (297.8501921 + 445267.1114034 * T - 0.0018819 * T**2 +
         T**3 / 545868 - T**4 / 113065000) % 360

    # Sun's mean anomaly (degrees)
    M = (357.5291092 + 35999.0502909 * T - 0.0001536 * T**2 +
         T**3 / 24490000) % 360

    # Moon's mean anomaly (degrees)
    Mp = (134.9633964 + 477198.8675055 * T + 0.0087414 * T**2 +
          T**3 / 69699 - T**4 / 14712000) % 360

    # Moon's argument of latitude (degrees)
    F = (93.2720950 + 483202.0175233 * T - 0.0036539 * T**2 -
         T**3 / 3526000 + T**4 / 863310000) % 360

    # Additional arguments
    A1 = (119.75 + 131.849 * T) % 360
    A2 = (53.09 + 479264.290 * T) % 360
    A3 = (313.45 + 481266.484 * T) % 360

    # Eccentricity correction
    E = 1 - 0.002516 * T - 0.0000074 * T**2

    # Convert to radians
    D_rad = math.radians(D)
    M_rad = math.radians(M)
    Mp_rad = math.radians(Mp)
    F_rad = math.radians(F)
    A1_rad = math.radians(A1)
    A2_rad = math.radians(A2)
    A3_rad = math.radians(A3)

    # Sum of longitude terms (most significant terms from Table 47.A)
    sum_l = (
        6288774 * math.sin(Mp_rad) +
        1274027 * math.sin(2*D_rad - Mp_rad) +
        658314 * math.sin(2*D_rad) +
        213618 * math.sin(2*Mp_rad) +
        -185116 * E * math.sin(M_rad) +
        -114332 * math.sin(2*F_rad) +
        58793 * math.sin(2*D_rad - 2*Mp_rad) +
        57066 * E * math.sin(2*D_rad - M_rad - Mp_rad) +
        53322 * math.sin(2*D_rad + Mp_rad) +
        45758 * E * math.sin(2*D_rad - M_rad) +
        -40923 * E * math.sin(M_rad - Mp_rad) +
        -34720 * math.sin(D_rad) +
        -30383 * E * math.sin(M_rad + Mp_rad) +
        15327 * math.sin(2*D_rad - 2*F_rad) +
        -12528 * math.sin(Mp_rad + 2*F_rad) +
        10980 * math.sin(Mp_rad - 2*F_rad) +
        10675 * math.sin(4*D_rad - Mp_rad) +
        10034 * math.sin(3*Mp_rad) +
        8548 * math.sin(4*D_rad - 2*Mp_rad) +
        -7888 * E * math.sin(2*D_rad + M_rad - Mp_rad) +
        -6766 * E * math.sin(2*D_rad + M_rad) +
        -5163 * math.sin(D_rad - Mp_rad) +
        4987 * E * math.sin(D_rad + M_rad) +
        4036 * E * math.sin(2*D_rad - M_rad + Mp_rad)
    )

    # Additional longitude corrections
    Lp_rad = math.radians(Lp)
    sum_l += (
        3958 * math.sin(A1_rad) +
        1962 * math.sin(Lp_rad - F_rad) +
        318 * math.sin(A2_rad)
    )

    # Sum of latitude terms (most significant from Table 47.B)
    sum_b = (
        5128122 * math.sin(F_rad) +
        280602 * math.sin(Mp_rad + F_rad) +
        277693 * math.sin(Mp_rad - F_rad) +
        173237 * math.sin(2*D_rad - F_rad) +
        55413 * math.sin(2*D_rad - Mp_rad + F_rad) +
        46271 * math.sin(2*D_rad - Mp_rad - F_rad) +
        32573 * math.sin(2*D_rad + F_rad) +
        17198 * math.sin(2*Mp_rad + F_rad) +
        9266 * math.sin(2*D_rad + Mp_rad - F_rad) +
        8822 * math.sin(2*Mp_rad - F_rad) +
        -8216 * E * math.sin(2*D_rad - M_rad - F_rad) +
        4324 * math.sin(2*D_rad - 2*Mp_rad - F_rad) +
        4200 * math.sin(2*D_rad + Mp_rad + F_rad) +
        -3359 * E * math.sin(2*D_rad + M_rad - F_rad) +
        2463 * E * math.sin(2*D_rad - M_rad - Mp_rad + F_rad) +
        2211 * E * math.sin(2*D_rad - M_rad + F_rad) +
        2065 * E * math.sin(2*D_rad - M_rad - Mp_rad - F_rad) +
        -1870 * E * math.sin(M_rad - Mp_rad - F_rad)
    )

    # Additional latitude corrections
    sum_b += (
        -2235 * math.sin(Lp_rad) +
        382 * math.sin(A3_rad) +
        175 * math.sin(A1_rad - F_rad) +
        175 * math.sin(A1_rad + F_rad) +
        127 * math.sin(Lp_rad - Mp_rad) +
        -115 * math.sin(Lp_rad + Mp_rad)
    )

    # Ecliptic longitude and latitude (degrees)
    ecl_lon = Lp + sum_l / 1000000
    ecl_lat = sum_b / 1000000

    ecl_lon_rad = math.radians(ecl_lon)
    ecl_lat_rad = math.radians(ecl_lat)

    # Mean obliquity of the ecliptic
    eps = 23.439291 - 0.0130042 * T
    eps_rad = math.radians(eps)

    # Convert ecliptic to equatorial
    x_ecl = math.cos(ecl_lat_rad) * math.cos(ecl_lon_rad)
    y_ecl = math.cos(ecl_lat_rad) * math.sin(ecl_lon_rad)
    z_ecl = math.sin(ecl_lat_rad)

    x_eq = x_ecl
    y_eq = y_ecl * math.cos(eps_rad) - z_ecl * math.sin(eps_rad)
    z_eq = y_ecl * math.sin(eps_rad) + z_ecl * math.cos(eps_rad)

    ra_rad = math.atan2(y_eq, x_eq)
    dec_rad = math.asin(z_eq)

    ra_hours = (math.degrees(ra_rad) / 15) % 24
    dec_deg = math.degrees(dec_rad)

    return ra_hours, dec_deg

In [ ]:
def ra_dec_to_alt_az(ra_hours, dec_deg, lat_deg, lon_deg, dt):
    """
    Convert RA/Dec to Alt/Az for a given datetime and location.
    """
    jd = julian_date(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second)

    # Calculate hour angle
    lst = local_sidereal_time(jd, lon_deg)
    ha_hours = lst - ra_hours
    ha_rad = math.radians(ha_hours * 15)

    # Convert to radians
    dec_rad = math.radians(dec_deg)
    lat_rad = math.radians(lat_deg)

    # Calculate altitude
    sin_alt = (math.sin(dec_rad) * math.sin(lat_rad) +
               math.cos(dec_rad) * math.cos(lat_rad) * math.cos(ha_rad))
    alt_rad = math.asin(sin_alt)

    # Calculate azimuth
    cos_az = ((math.sin(dec_rad) - math.sin(alt_rad) * math.sin(lat_rad)) /
              (math.cos(alt_rad) * math.cos(lat_rad)))
    cos_az = max(-1, min(1, cos_az))
    az_rad = math.acos(cos_az)

    # Adjust azimuth quadrant
    if math.sin(ha_rad) > 0:
        az_rad = 2 * math.pi - az_rad

    alt_deg = math.degrees(alt_rad)
    az_deg = math.degrees(az_rad)

    return alt_deg, az_deg

In [ ]:
def galactic_to_equatorial(l_deg, b_deg):
    """
    Convert Galactic coordinates to Equatorial (J2000).
    """
    # Galactic coordinate system constants (J2000)
    ra_ngp = math.radians(192.85948)
    dec_ngp = math.radians(27.12825)
    l_ncp = math.radians(122.93192)

    l_rad = math.radians(l_deg)
    b_rad = math.radians(b_deg)

    # Calculate declination
    sin_dec = (math.sin(dec_ngp) * math.sin(b_rad) +
               math.cos(dec_ngp) * math.cos(b_rad) * math.cos(l_ncp - l_rad))
    dec_rad = math.asin(sin_dec)

    # Calculate right ascension
    y = math.cos(b_rad) * math.sin(l_ncp - l_rad)
    x = (math.cos(dec_ngp) * math.sin(b_rad) -
         math.sin(dec_ngp) * math.cos(b_rad) * math.cos(l_ncp - l_rad))

    ra_rad = ra_ngp + math.atan2(y, x)

    ra_hours = (math.degrees(ra_rad) / 15) % 24
    dec_deg = math.degrees(dec_rad)

    return ra_hours, dec_deg

## Test 1: Sun Position

In [ ]:
# Test dates spanning different seasons
test_dates = [
    datetime(2024, 3, 20, 12, 0, 0, tzinfo=timezone.utc),  # Spring equinox
    datetime(2024, 6, 21, 12, 0, 0, tzinfo=timezone.utc),  # Summer solstice
    datetime(2024, 9, 22, 12, 0, 0, tzinfo=timezone.utc),  # Autumn equinox
    datetime(2024, 12, 21, 12, 0, 0, tzinfo=timezone.utc), # Winter solstice
    datetime(2026, 3, 13, 14, 30, 0, tzinfo=timezone.utc), # Current date
]

print("Sun Position Comparison")
print("=" * 80)
print(f"{'Date':<22} {'SRT RA':>10} {'Astropy RA':>12} {'RA Err':>10} {'SRT Dec':>10} {'Astropy Dec':>12} {'Dec Err':>10}")
print("-" * 80)

sun_ra_errors = []
sun_dec_errors = []

for dt in test_dates:
    # SRT calculation
    srt_ra, srt_dec = get_sun_position(dt)
    
    # Astropy calculation
    t = Time(dt)
    sun = get_sun(t)
    astropy_ra = sun.ra.hour
    astropy_dec = sun.dec.deg
    
    # Calculate errors (handle RA wrap-around)
    ra_err = srt_ra - astropy_ra
    if ra_err > 12:
        ra_err -= 24
    elif ra_err < -12:
        ra_err += 24
    ra_err_arcmin = ra_err * 15 * 60  # Convert hours to arcminutes
    
    dec_err = srt_dec - astropy_dec
    dec_err_arcmin = dec_err * 60  # Convert degrees to arcminutes
    
    sun_ra_errors.append(abs(ra_err_arcmin))
    sun_dec_errors.append(abs(dec_err_arcmin))
    
    print(f"{dt.strftime('%Y-%m-%d %H:%M'):<22} {srt_ra:>10.4f}h {astropy_ra:>10.4f}h {ra_err_arcmin:>+9.2f}' {srt_dec:>+10.4f}° {astropy_dec:>+10.4f}° {dec_err_arcmin:>+9.2f}'")

print("-" * 80)
print(f"Mean absolute error: RA = {np.mean(sun_ra_errors):.2f} arcmin, Dec = {np.mean(sun_dec_errors):.2f} arcmin")
print(f"Max absolute error:  RA = {np.max(sun_ra_errors):.2f} arcmin, Dec = {np.max(sun_dec_errors):.2f} arcmin")

## Test 2: Moon Position

In [ ]:
# Test dates for Moon (different lunar phases)
moon_test_dates = [
    datetime(2024, 1, 11, 12, 0, 0, tzinfo=timezone.utc),  # New moon
    datetime(2024, 1, 18, 12, 0, 0, tzinfo=timezone.utc),  # First quarter
    datetime(2024, 1, 25, 12, 0, 0, tzinfo=timezone.utc),  # Full moon
    datetime(2024, 2, 2, 12, 0, 0, tzinfo=timezone.utc),   # Last quarter
    datetime(2026, 3, 13, 14, 30, 0, tzinfo=timezone.utc), # Current date
    datetime(2024, 6, 15, 6, 0, 0, tzinfo=timezone.utc),   # Random date 1
    datetime(2024, 9, 20, 18, 0, 0, tzinfo=timezone.utc),  # Random date 2
]

print("Moon Position Comparison")
print("=" * 80)
print(f"{'Date':<22} {'SRT RA':>10} {'Astropy RA':>12} {'RA Err':>10} {'SRT Dec':>10} {'Astropy Dec':>12} {'Dec Err':>10}")
print("-" * 80)

moon_ra_errors = []
moon_dec_errors = []

for dt in moon_test_dates:
    # SRT calculation
    srt_ra, srt_dec = get_moon_position(dt)
    
    # Astropy calculation
    t = Time(dt)
    moon = get_body('moon', t)
    astropy_ra = moon.ra.hour
    astropy_dec = moon.dec.deg
    
    # Calculate errors
    ra_err = srt_ra - astropy_ra
    if ra_err > 12:
        ra_err -= 24
    elif ra_err < -12:
        ra_err += 24
    ra_err_arcmin = ra_err * 15 * 60
    
    dec_err = srt_dec - astropy_dec
    dec_err_arcmin = dec_err * 60
    
    moon_ra_errors.append(abs(ra_err_arcmin))
    moon_dec_errors.append(abs(dec_err_arcmin))
    
    print(f"{dt.strftime('%Y-%m-%d %H:%M'):<22} {srt_ra:>10.4f}h {astropy_ra:>10.4f}h {ra_err_arcmin:>+9.2f}' {srt_dec:>+10.4f}° {astropy_dec:>+10.4f}° {dec_err_arcmin:>+9.2f}'")

print("-" * 80)
print(f"Mean absolute error: RA = {np.mean(moon_ra_errors):.2f} arcmin, Dec = {np.mean(moon_dec_errors):.2f} arcmin")
print(f"Max absolute error:  RA = {np.max(moon_ra_errors):.2f} arcmin, Dec = {np.max(moon_dec_errors):.2f} arcmin")

## Test 3: RA/Dec to Alt/Az Conversion

In [ ]:
# Acre Road Observatory location
OBSERVER_LAT = 55.9
OBSERVER_LON = -4.3

location = EarthLocation(lat=OBSERVER_LAT*u.deg, lon=OBSERVER_LON*u.deg, height=50*u.m)

# Test objects at different positions
test_objects = [
    ("Polaris", 2.53, 89.26),           # Near NCP
    ("Vega", 18.62, 38.78),             # Bright star
    ("Betelgeuse", 5.92, 7.41),         # Near equator
    ("Sirius", 6.75, -16.72),           # Southern
    ("Galactic Center", 17.76, -29.0),  # Sgr A*
]

dt = datetime(2026, 3, 13, 20, 0, 0, tzinfo=timezone.utc)  # Evening observation
t = Time(dt)

print(f"Alt/Az Conversion Test at {dt.strftime('%Y-%m-%d %H:%M')} UTC")
print(f"Location: {OBSERVER_LAT}°N, {OBSERVER_LON}°E")
print("=" * 90)
print(f"{'Object':<18} {'SRT Alt':>10} {'Astropy Alt':>12} {'Alt Err':>10} {'SRT Az':>10} {'Astropy Az':>12} {'Az Err':>10}")
print("-" * 90)

alt_errors = []
az_errors = []

for name, ra, dec in test_objects:
    # SRT calculation
    srt_alt, srt_az = ra_dec_to_alt_az(ra, dec, OBSERVER_LAT, OBSERVER_LON, dt)
    
    # Astropy calculation
    coord = SkyCoord(ra=ra*u.hourangle, dec=dec*u.deg, frame='icrs')
    altaz = coord.transform_to(AltAz(obstime=t, location=location))
    astropy_alt = altaz.alt.deg
    astropy_az = altaz.az.deg
    
    alt_err = (srt_alt - astropy_alt) * 60  # arcmin
    az_err = srt_az - astropy_az
    if az_err > 180:
        az_err -= 360
    elif az_err < -180:
        az_err += 360
    az_err *= 60  # arcmin
    
    alt_errors.append(abs(alt_err))
    az_errors.append(abs(az_err))
    
    print(f"{name:<18} {srt_alt:>+10.3f}° {astropy_alt:>+10.3f}° {alt_err:>+9.2f}' {srt_az:>10.3f}° {astropy_az:>10.3f}° {az_err:>+9.2f}'")

print("-" * 90)
print(f"Mean absolute error: Alt = {np.mean(alt_errors):.2f} arcmin, Az = {np.mean(az_errors):.2f} arcmin")
print(f"Max absolute error:  Alt = {np.max(alt_errors):.2f} arcmin, Az = {np.max(az_errors):.2f} arcmin")

## Test 4: Galactic to Equatorial Conversion

In [ ]:
# Test galactic coordinates
galactic_tests = [
    ("Galactic Center", 0.0, 0.0),
    ("Galactic Anticenter", 180.0, 0.0),
    ("North Galactic Pole", 0.0, 90.0),
    ("South Galactic Pole", 0.0, -90.0),
    ("Cygnus X", 80.0, 0.0),
    ("Cas A region", 111.7, -2.1),
]

print("Galactic to Equatorial Conversion Test")
print("=" * 90)
print(f"{'Object':<20} {'l':>8} {'b':>8} {'SRT RA':>10} {'Astropy RA':>12} {'SRT Dec':>10} {'Astropy Dec':>12}")
print("-" * 90)

gal_ra_errors = []
gal_dec_errors = []

for name, l, b in galactic_tests:
    # SRT calculation
    srt_ra, srt_dec = galactic_to_equatorial(l, b)
    
    # Astropy calculation
    coord = SkyCoord(l=l*u.deg, b=b*u.deg, frame='galactic')
    icrs = coord.icrs
    astropy_ra = icrs.ra.hour
    astropy_dec = icrs.dec.deg
    
    ra_err = srt_ra - astropy_ra
    if ra_err > 12:
        ra_err -= 24
    elif ra_err < -12:
        ra_err += 24
    ra_err_arcmin = ra_err * 15 * 60
    
    dec_err = (srt_dec - astropy_dec) * 60
    
    gal_ra_errors.append(abs(ra_err_arcmin))
    gal_dec_errors.append(abs(dec_err))
    
    print(f"{name:<20} {l:>8.1f}° {b:>+8.1f}° {srt_ra:>10.4f}h {astropy_ra:>10.4f}h {srt_dec:>+10.4f}° {astropy_dec:>+10.4f}°")

print("-" * 90)
print(f"Mean absolute error: RA = {np.mean(gal_ra_errors):.4f} arcmin, Dec = {np.mean(gal_dec_errors):.4f} arcmin")

## Test 5: Julian Date Calculation

In [ ]:
# Test Julian Date calculation against known values
jd_tests = [
    ((2000, 1, 1, 12, 0, 0), 2451545.0, "J2000.0 epoch"),
    ((1858, 11, 17, 0, 0, 0), 2400000.5, "MJD epoch"),
    ((2024, 3, 20, 3, 6, 0), 2460389.62917, "Vernal equinox 2024"),
]

print("Julian Date Calculation Test")
print("=" * 70)
print(f"{'Description':<25} {'Calculated JD':>18} {'Expected JD':>18} {'Error':>10}")
print("-" * 70)

for (y, m, d, h, mi, s), expected, desc in jd_tests:
    calculated = julian_date(y, m, d, h, mi, s)
    error = calculated - expected
    print(f"{desc:<25} {calculated:>18.5f} {expected:>18.5f} {error:>+10.6f}")

## Summary

In [ ]:
print("\n" + "=" * 60)
print("ACCURACY SUMMARY")
print("=" * 60)
print(f"\nSun Position:")
print(f"  Mean error: {np.mean(sun_ra_errors):.2f}' RA, {np.mean(sun_dec_errors):.2f}' Dec")
print(f"  Max error:  {np.max(sun_ra_errors):.2f}' RA, {np.max(sun_dec_errors):.2f}' Dec")

print(f"\nMoon Position:")
print(f"  Mean error: {np.mean(moon_ra_errors):.2f}' RA, {np.mean(moon_dec_errors):.2f}' Dec")
print(f"  Max error:  {np.max(moon_ra_errors):.2f}' RA, {np.max(moon_dec_errors):.2f}' Dec")

print(f"\nAlt/Az Conversion:")
print(f"  Mean error: {np.mean(alt_errors):.2f}' Alt, {np.mean(az_errors):.2f}' Az")
print(f"  Max error:  {np.max(alt_errors):.2f}' Alt, {np.max(az_errors):.2f}' Az")

print(f"\nGalactic Conversion:")
print(f"  Mean error: {np.mean(gal_ra_errors):.4f}' RA, {np.mean(gal_dec_errors):.4f}' Dec")

# Typical SRT beam width at 1420 MHz
beam_width = 7 * 60  # arcmin (assuming ~7 degree beam)
print(f"\n" + "-" * 60)
print(f"Typical SRT beam width at 1420 MHz: ~{beam_width/60:.0f}° ({beam_width:.0f} arcmin)")
print(f"All errors are well within the beam width.")
print("=" * 60)